# Installations

In [ ]:
# installing offline dependencies
!pip install -U /kaggle/input/faiss-gpu-173-python310/faiss_gpu-1.7.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!cp -rf /kaggle/input/sentence-transformers-222/sentence-transformers /kaggle/working/sentence-transformers
!pip install -U /kaggle/working/sentence-transformers
!pip install -U /kaggle/input/blingfire-018/blingfire-0.1.8-py3-none-any.whl

!pip install --no-index --no-deps /kaggle/input/llm-whls/transformers-4.31.0-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/peft-0.4.0-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/datasets-2.14.3-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/trl-0.5.0-py3-none-any.whl

# Load libraries

In [ ]:
import os, time
import gc
import pandas as pd
import numpy as np
import re
from tqdm.auto import tqdm
import blingfire as bf
from __future__ import annotations

from collections.abc import Iterable

import faiss
from faiss import write_index, read_index

from sentence_transformers import SentenceTransformer

import torch
import ctypes
libc = ctypes.CDLL("libc.so.6")

from dataclasses import dataclass
from typing import Optional, Union

import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from torch.utils.data import DataLoader

from scipy.special import softmax

In [ ]:
from scipy.optimize import minimize, fsolve
import datetime
import torch.nn.functional as F
from numba import njit

In [ ]:
SIM_MODEL = '/kaggle/input/sentencetransformers-allminilml6v2/sentence-transformers_all-MiniLM-L6-v2'
DEVICE = 0
MAX_LENGTH = 384
BATCH_SIZE = 32

trn = pd.read_csv("/kaggle/input/kaggle-llm-science-exam/test.csv").drop("id", axis=1)

DEBUG = False
# DEBUG = False if len(trn)!=200 else True # If you want to save GPU Quota, check off this comment-out. But cannot get accurate weight on saving notebook
FILTER_LEN = 1 if DEBUG else 10
IND_SEARCH = 1 if DEBUG else 7
NUM_SENTENCES_INCLUDE = 1 if DEBUG else 22
CONTEXT_LEN = 1000 if DEBUG else 2305
VAL_SIZE = 200 if DEBUG else 1500

WIKI_PATH = "/kaggle/input/wikipedia-20230701"
wiki_files = os.listdir(WIKI_PATH)

# Custom functions

In [ ]:
def process_documents(documents: Iterable[str],
                      document_ids: Iterable,
                      split_sentences: bool = True,
                      filter_len: int = FILTER_LEN,
                      disable_progress_bar: bool = False) -> pd.DataFrame:
    
    df = sectionize_documents(documents, document_ids, disable_progress_bar)

    if split_sentences:
        df = sentencize(df.text.values, 
                        df.document_id.values,
                        df.offset.values, 
                        filter_len, 
                        disable_progress_bar)
    return df


def sectionize_documents(documents: Iterable[str],
                         document_ids: Iterable,
                         disable_progress_bar: bool = False) -> pd.DataFrame:

    processed_documents = []
    for document_id, document in tqdm(zip(document_ids, documents), total=len(documents), disable=disable_progress_bar):
        row = {}
        text, start, end = (document, 0, len(document))
        row['document_id'] = document_id
        row['text'] = text
        row['offset'] = (start, end)

        processed_documents.append(row)

    _df = pd.DataFrame(processed_documents)
    if _df.shape[0] > 0:
        return _df.sort_values(['document_id', 'offset']).reset_index(drop=True)
    else:
        return _df


def sentencize(documents: Iterable[str],
               document_ids: Iterable,
               offsets: Iterable[tuple[int, int]],
               filter_len: int = FILTER_LEN,
               disable_progress_bar: bool = False) -> pd.DataFrame:

    document_sentences = []
    for document, document_id, offset in tqdm(zip(documents, document_ids, offsets), total=len(documents), disable=disable_progress_bar):
        try:
            _, sentence_offsets = bf.text_to_sentences_and_offsets(document)
            for o in sentence_offsets:
                if o[1]-o[0] > filter_len:
                    sentence = document[o[0]:o[1]]
                    abs_offsets = (o[0]+offset[0], o[1]+offset[0])
                    row = {}
                    row['document_id'] = document_id
                    row['text'] = sentence
                    row['offset'] = abs_offsets
                    document_sentences.append(row)
        except:
            continue
    return pd.DataFrame(document_sentences)

In [ ]:
def pickle_dump(path, saveobj):
    import pickle
    filehandler = open(path,"wb")
    pickle.dump(saveobj,filehandler)
    print("File pickled")
    filehandler.close()



def pickle_load(path):
    import pickle
    file = open(path,'rb')
    loadobj = pickle.load(file)
    file.close()
    return loadobj

In [ ]:
def apk(actual, predicted, k=5):
    """
    Computes the average precision at k.
    This function computes the average prescision at k between two lists of
    items.
    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int, optional
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The average precision at k over the input lists
    """
    
    # requires all elements are unique
    assert (len(np.unique(predicted)) == len(predicted))

    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        # first condition checks whether it is valid prediction
        # second condition checks if prediction is not repeated
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    return score / min(len(actual), k)

def mapk(actual, predicted, k=5):
    
    """
    Computes the mean average precision at k.
    This function computes the mean average prescision at k between two lists
    of lists of items.
    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted 
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int, optional
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The mean average precision at k over the input lists
    """
    return np.mean([apk(a,p,k) for a,p in zip(actual, predicted)])

In [ ]:
@njit
def grad_func_jit(weights):
    preds_clip = np.minimum(1 - 1e-15, np.maximum(preds, 1e-15))
    gradients = np.zeros(preds.shape[0])
    for i in range(preds.shape[0]):
        a, b, c = target_values, preds_clip[i], np.zeros((preds.shape[1], preds.shape[2]))
        a = np.eye(5)[a]
        for j in range(preds.shape[0]):
            if j != i:
                c += weights[j] * preds_clip[j]
        gradients[i] = -np.mean((-a*b+(b**2)*weights[i]+b*c)/((b**2)*(weights[i]**2)+2*b*c*weights[i]-b*weights[i]+(c**2)-c))
    return gradients

In [ ]:
def calc_mtr(predicted, k=3):
    y_preds = np.argsort(-predicted, 1)
    map3 = mapk(target_values.reshape(-1, 1), y_preds.reshape(-1, 5), k=k)
    return map3

# def calc_loss(predicted):
#     score = F.cross_entropy(torch.tensor(predicted), torch.tensor(target_values)).numpy()
#     return score


def log_loss_numpy(y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    loss = - target_values_one_hot * np.log(y_pred)
    loss = np.sum(loss, axis=-1)
    return loss.mean()

def func_to_optimise(weights):
    pred_blend = np.tensordot(weights, preds, axes = ((0), (0)))
    score = log_loss_numpy(pred_blend)
    return score

def func_to_map3(weights):
    pred_blend = np.tensordot(weights, preds, axes = ((0), (0)))
    score = calc_mtr(pred_blend)
    return score

# Read data

In [ ]:
val_df = pd.read_csv('/kaggle/input/mmlu-dataset-valid-only/valid_mmlu_1526_ind0.csv',index_col=0)[:VAL_SIZE]

val_df['E'] = '' # dummy answer that allows us to preprocess the test datataset using functionality that works for the train set
val_df = val_df.replace(np.NaN, '')

val_df['A'] = val_df['A'].map(str)
val_df['B'] = val_df['B'].map(str)
val_df['C'] = val_df['C'].map(str)
val_df['D'] = val_df['D'].map(str)
val_df['E'] = val_df['E'].map(str)

val_df.reset_index(inplace=True, drop=True)

val_df.shape

In [ ]:
options = 'ABCDE'
indices = list(range(5))

option_to_index = {option: index for option, index in zip(options, indices)}
index_to_option = {index: option for option, index in zip(options, indices)}
target_values = val_df['answer'].map(option_to_index).values
target_values_one_hot = np.eye(5)[target_values]

# Get Val prediction arrays

In [ ]:
# preds_dict = {
#     'chris': val_predictionsc,
#     'openbook': ob_preds_v,
#     'itk_ob': val_predictionsi,   
#     'hyc': hyc_preds_v,
#     'itk_awp': itk_preds_v
# }

In [ ]:
VAL_LOC = "/kaggle/input/llm-sciex-optimise-ensemble-weights-better-models/"

chris_df = pd.read_csv(VAL_LOC + "chris_val.csv")
openbook_df = pd.read_csv(VAL_LOC + "openbook_val.csv")
itkob_df = pd.read_csv(VAL_LOC + "itk_ob_val.csv")
hyc_df = pd.read_csv(VAL_LOC + "hyc_val.csv")
itkawp_df = pd.read_csv(VAL_LOC + "itk_awp_val.csv")
illi_df = pd.read_csv("/kaggle/input/llmse-mmluvalpreds/illimodel1_val.csv")
illi2_df = pd.read_csv("/kaggle/input/llmse-mmluvalpreds/Debertav3Large-60k-256-checkpoint-1025_mmlu_val.csv")
illi3_df = pd.read_csv("/kaggle/input/llmse-mmluvalpreds/Debertav3Large-30k-256-checkpoint-200_mmlu_val.csv")
illi4_df = pd.read_csv("/kaggle/input/llmse-mmluvalpreds/Debertav3Large-30k-256-checkpoint-700_mmlu_val.csv")


print("Chris: ", chris_df.shape)
print("Openbook: ", openbook_df.shape)
print("ITK OB: ", itkob_df.shape)
print("HYC: ", hyc_df.shape)
print("ITK AWP: ", itkawp_df.shape)
print("Illimodel1: ", illi_df.shape)
print("Illimodel2: ", illi2_df.shape)
print("Illimodel3: ", illi3_df.shape)
print("Illimodel4: ", illi4_df.shape)

# Prediction scores

In [ ]:
keepCols = ['A_prob', 'B_prob', 'C_prob', 'D_prob', 'E_prob']

preds_dict = {
    'chris': np.array(chris_df[keepCols]),
#     'openbook': np.array(openbook_df[keepCols]),
#     'itk_ob': np.array(itkob_df[keepCols]),   
#     'hyc': np.array(hyc_df[keepCols]),
    'itk_awp': np.array(itkawp_df[keepCols]),
    'illi1': np.array(illi_df[keepCols]),
#     'illi2': np.array(illi2_df[keepCols]),
#     'illi3': np.array(illi3_df[keepCols]),
    'illi4': np.array(illi4_df[keepCols])
}

In [ ]:
preds = np.zeros((len(preds_dict), len(val_df), 5))
for i in range(preds.shape[0]):
    preds[i] = list(preds_dict.values())[i]

In [ ]:
%%time

map3_scores = {}
for n, key in enumerate(preds_dict.keys()):
    score_val = calc_mtr(preds[n])
    map3_scores[key] = score_val
    print(f'{key:40s} CV_map@3:', score_val)
    
print('-' * 60)

loss_scores = {}
for n, key in enumerate(preds_dict.keys()):
    score_val = log_loss_numpy(preds[n])
    loss_scores[key] = score_val
    print(f'{key:40s} CV_CELoss:', score_val)
    
print('-' * 60)

ln(5) = 1.60943791243 and losses are about 1.5, So the models seem to be making near-random predictions. As I said at the beginning, the validation dataset may not be suitable, but I will continue.

### Observe correlation
It is not the good way to increase the weight of model just because the CV is high. Correlation between models is also an important index in determining weights. In general, we can aim for a high score improvement by ensemble models with good CV and low correlation.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
import seaborn as sns
from matplotlib import pyplot
from matplotlib.ticker import ScalarFormatter
sns.set_context("talk")
style.use('fivethirtyeight')

subs = np.zeros((len(preds_dict), len(val_df), 5))

for i, p in enumerate(preds_dict.keys()):
    print(i,p)
    subs[i,:,:] = list(preds_dict.values())[i]
    
corr = np.corrcoef(subs.reshape(len(preds_dict), -1))

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(15, 12))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, cmap=cmap, annot=True, fmt="g",
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, ax=ax)
ax.set_ylim(corr.shape[0], 0)
plt.yticks(rotation=0)

## Blending Weights Optimize

Maximising MAP@3 is very difficult(Is it even possible?). so Minimising CE loss here.

In [ ]:
tol = 1e-10
init_guess = [1 / preds.shape[0]] * preds.shape[0]
bnds = [(0, 1) for _ in range(preds.shape[0])]
cons = {'type': 'eq', 
        'fun': lambda x: np.sum(x) - 1, 
        'jac': lambda x: [1] * len(x)}

print('Inital Blend Loss:', func_to_optimise(init_guess))
print('Inital Blend MAP@3:', func_to_map3(init_guess))
start_time = time.time()

res_scipy = minimize(fun = func_to_optimise, 
                     x0 = init_guess, 
                     method = 'SLSQP', 
                     tol = tol,
                     bounds = bnds,
                     jac = grad_func_jit, 
                     constraints = cons,
                     options={"disp":True,"maxiter":1000})

print(f'[{str(datetime.timedelta(seconds = time.time() - start_time))[2:7]}] Optimised Blend Loss:', res_scipy.fun, ', Optimised Blend MAP@3:', func_to_map3(res_scipy.x))
print('Optimised Weights:', res_scipy.x)
print('-' * 70)

for n, key in enumerate(preds_dict.keys()):
    print(f'{key:40s} Optimised Weights:', res_scipy.x[n])

# Final weights

In [ ]:
ws = [res_scipy.x[i] for i in range(len(preds_dict.keys()))]
ws = ws / np.sum(ws)
ws

In [ ]:
for n, key in enumerate(preds_dict.keys()):
    print(f'{key:40s} Optimised Weights:', res_scipy.x[n])

In [ ]:
predictions_overall = predictions_overall
predictions_overall = np.argsort(-predictions_overall)[:,:3]
predictions_overall[:5]

In [ ]:
predictions_as_answer_letters = np.array(list('ABCDE'))[predictions_overall]
predictions_as_answer_letters[:3]

In [ ]:
predictions_as_string = test_df['prediction'] = [
    ' '.join(row) for row in predictions_as_answer_letters[:, :3]
]
predictions_as_string[:3]

In [ ]:
submission = test_df[['id', 'prediction']]
submission.to_csv('submission.csv', index=False)

pd.read_csv('submission.csv').head(10)

In conclusion, at least we were able to confirm that the openbook model (based on Ozturk's and Chris'), which differs in method from other models and has a high score, has the higher weight.

Now it's your turn to blend. Let's add weights for your model. 

Also, running notebooks, especially inference for openbook model, takes a long time, so it's a good idea to separate notebooks for calculating weights and for submitting them like Yirun Zhangs' base notebook.

It would also be important to change the evaluation dataset to something relevant to STEM. If the model weights are unnaturally high, suspect a leak. And make sure the evaluation dataset is not used for training.

### Wishing you happy kaggling!